# Water-Net Inference & Benchmarking Pipeline (Google Colab)

This notebook runs **end-to-end inference and evaluation** for the official **Water-Net**
underwater image enhancement model (Li et al., IEEE TIP 2019), using the Python
preprocessing pipeline contributed by **Branimir Ambrekovic** (white balance, gamma
correction, and histogram equalization implemented in Python/OpenCV, so **no MATLAB is
required**).

**Repository:** https://github.com/Li-Chongyi/Water-Net_Code
**Preprocessing code used:** `testing_code_by_Branimir Ambrekovic/`

### ⚠️ Important naming clarification (read this first)

The official Water-Net repo happens to use the folder names `gc_real` and `ce_real` for
its **own internal preprocessing outputs** (gamma-corrected images and histogram-equalized
images respectively).

**Your dataset uses the same folder names for something completely different:**

| Your folder | Meaning in YOUR dataset |
|---|---|
| `Dataset/gc_real` | Raw / degraded underwater images (network **input**) |
| `Dataset/ce_real`  | Ground-truth reference images (used only for **evaluation**) |

To avoid *any* collision between these two unrelated meanings, this notebook:
- Never writes into folders literally called `gc_real` / `ce_real` for Water-Net's own
  preprocessing. Instead it creates clearly-named working folders
  `_wn_work/wb_input`, `_wn_work/gc_input`, `_wn_work/ce_input` for the network's
  white-balance / gamma-corrected / histogram-equalized inputs.
- Always refers to your dataset explicitly as **raw** (`Dataset/gc_real`) and
  **ground truth** (`Dataset/ce_real`).

### What this notebook does
1. Clones the repo and installs dependencies.
2. Mounts your Google Drive and verifies the checkpoint.
3. Loads your raw images from `MyDrive/Dataset/gc_real` and reproduces, in pure Python,
   the white-balance / gamma-correction / histogram-equalization preprocessing that
   Water-Net requires (no MATLAB).
4. Rebuilds the exact Water-Net TensorFlow graph (auto-upgraded to run under TensorFlow
   2.x via `tf.compat.v1` + `disable_eager_execution`) and restores your pretrained
   `coarse_112` checkpoint.
5. Runs batched inference over the whole dataset with a progress bar, saving results to
   `MyDrive/Results/WaterNet_Output/`.
6. Visualizes Raw → Enhanced → Ground-Truth triplets.
7. Evaluates PSNR, SSIM, UIQM, UCIQE against `MyDrive/Dataset/ce_real`, exports
   `metrics.csv` / `metrics.xlsx`.
8. Prints a final summary.

Run the cells **in order, top to bottom**. Nothing needs to be edited manually.


## Section 2 — Clone the official Water-Net repository

In [ ]:
#@title Clone Water-Net repository
import os

REPO_DIR = "/content/Water-Net_Code"

if not os.path.exists(REPO_DIR):
    !git clone --quiet https://github.com/Li-Chongyi/Water-Net_Code.git "{REPO_DIR}"
else:
    print("Repository already present, skipping clone.")

BRANIMIR_DIR = os.path.join(REPO_DIR, "testing_code_by_Branimir Ambrekovic")

assert os.path.isdir(REPO_DIR), "Clone failed — repository directory not found."
print("Repository ready at:", REPO_DIR)
if os.path.isdir(BRANIMIR_DIR):
    print("Branimir Ambrekovic Python preprocessing folder found:", BRANIMIR_DIR)
else:
    print("NOTE: Branimir's folder was not found in the clone; this notebook ships its "
          "own equivalent Python implementation of the same preprocessing (white "
          "balance / gamma correction / histogram equalization), so this is not fatal.")


## Section 3 — Install dependencies

We pin to TensorFlow 2.x (Colab's default) and run the original TF1-style Water-Net graph
through the `tf.compat.v1` compatibility layer with eager execution disabled — this is
handled automatically in Section 7, no manual code changes needed.

In [ ]:
#@title Install/upgrade all required packages
import sys

packages = [
    "tensorflow>=2.12,<2.17",
    "opencv-python-headless",
    "Pillow",
    "numpy<2.0",
    "scikit-image",
    "matplotlib",
    "tqdm",
    "imageio",
    "pandas",
    "openpyxl",
    "gdown",
]

!{sys.executable} -m pip install -q {" ".join(packages)}

import tensorflow as tf
print("TensorFlow version:", tf.__version__)


In [ ]:
#@title Core imports used throughout the notebook
import os
import glob
import shutil
import math
import time
import traceback

import numpy as np
import cv2
import imageio
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import tensorflow as tf
tf.compat.v1.disable_eager_execution()   # run the original TF1-style graph under TF2

from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim

print("All core libraries imported successfully.")
print("GPU available:", tf.config.list_physical_devices('GPU'))


## Section 4 — Mount Google Drive

In [ ]:
#@title Mount Google Drive
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive"
assert os.path.isdir(DRIVE_ROOT), "Google Drive did not mount correctly."
print("Google Drive mounted at:", DRIVE_ROOT)


In [ ]:
#@title Configure all project paths (edit ONLY if your folder names differ)
CHECKPOINT_DIR   = os.path.join(DRIVE_ROOT, "WaterNet", "checkpoint")   # contains coarse_112/
DATASET_DIR      = os.path.join(DRIVE_ROOT, "Dataset")
RAW_DIR          = os.path.join(DATASET_DIR, "gc_real")   # YOUR raw underwater images
GT_DIR           = os.path.join(DATASET_DIR, "ce_real")   # YOUR ground-truth images
RESULTS_DIR      = os.path.join(DRIVE_ROOT, "Results", "WaterNet_Output")

# Fallback Google Drive shared-folder links (used automatically only if the paths above
# are missing / empty, so you never have to do anything manually).
CHECKPOINT_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/12PvwAgLWaqH9s5jSz-yCmu8lx2Icw5B_"
DATASET_DRIVE_FOLDER_URL    = "https://drive.google.com/drive/folders/1-6XfkD1hXcZdT3agKS979lUwwTuJmMnp"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Checkpoint dir :", CHECKPOINT_DIR)
print("Raw images dir :", RAW_DIR)
print("GT images dir  :", GT_DIR)
print("Results dir    :", RESULTS_DIR)


## Section 5 — Verify and locate the Water-Net checkpoint

Expected structure (as you already have it):
```
MyDrive/WaterNet/checkpoint/coarse_112/
    checkpoint
    coarse.model-1500.data-00000-of-00001
    coarse.model-1500.index
    coarse.model-1500.meta
```
`coarse_112` is not an arbitrary name — Water-Net's own checkpoint loader builds the
folder name as `"coarse_%s" % label_height` with `label_height=112`, so this notebook
looks specifically inside a `coarse_112` subfolder, matching your checkpoint exactly.

In [ ]:
#@title Verify checkpoint exists (auto-download from Drive link as a fallback)
import gdown

COARSE_DIR = os.path.join(CHECKPOINT_DIR, "coarse_112")

def checkpoint_is_valid(coarse_dir):
    if not os.path.isdir(coarse_dir):
        return False
    has_index = len(glob.glob(os.path.join(coarse_dir, "coarse.model-*.index"))) > 0
    has_data  = len(glob.glob(os.path.join(coarse_dir, "coarse.model-*.data-*"))) > 0
    has_ckpt_file = os.path.isfile(os.path.join(coarse_dir, "checkpoint"))
    return has_index and has_data and has_ckpt_file

if not checkpoint_is_valid(COARSE_DIR):
    print("Checkpoint not found at expected path — attempting automatic download from "
          "the shared Google Drive folder as a fallback...")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    try:
        gdown.download_folder(url=CHECKPOINT_DRIVE_FOLDER_URL, output=CHECKPOINT_DIR,
                               quiet=False, use_cookies=False)
    except Exception as e:
        print("Automatic download failed:", e)

if not checkpoint_is_valid(COARSE_DIR):
    raise FileNotFoundError(
        f"Could not find a valid Water-Net checkpoint in '{COARSE_DIR}'.\n"
        f"Expected a 'checkpoint' index file plus 'coarse.model-*.index' / "
        f"'coarse.model-*.data-*' files.\n"
        f"Please make sure your checkpoint is placed at:\n"
        f"  MyDrive/WaterNet/checkpoint/coarse_112/\n"
        f"or that the shared Drive folder link is accessible to your account."
    )

print("Checkpoint verified successfully:")
for f in sorted(os.listdir(COARSE_DIR)):
    print("  -", f)


## Section 6 — Prepare the dataset (automatic Python preprocessing, no MATLAB)

Water-Net needs **four** aligned inputs per image: the raw image plus three
pre-processed variants — **white-balanced**, **gamma-corrected**, and
**histogram-equalized**. These are generated automatically below in pure Python
(OpenCV / NumPy), following the same white-balance and gamma-correction routines
shipped in `testing_code_by_Branimir Ambrekovic/utils.py`, plus a standard luminance
histogram-equalization step for contrast enhancement — **no MATLAB, no manual folder
creation.**

Outputs are written to `_wn_work/` (a scratch directory), never to anything named
`gc_real` / `ce_real`, to avoid clashing with your dataset's own folders of those names.

In [ ]:
#@title Verify dataset folders and list matching raw/GT image pairs
def list_images(folder):
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.PNG", "*.JPG", "*.JPEG", "*.BMP")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(folder, e)))
    return sorted(files)

if not os.path.isdir(RAW_DIR):
    raise FileNotFoundError(f"Raw image folder not found: {RAW_DIR}")
if not os.path.isdir(GT_DIR):
    raise FileNotFoundError(f"Ground-truth folder not found: {GT_DIR}")

raw_files = list_images(RAW_DIR)
gt_files  = list_images(GT_DIR)

if len(raw_files) == 0:
    raise FileNotFoundError(f"No images found in raw folder: {RAW_DIR}")
if len(gt_files) == 0:
    raise FileNotFoundError(f"No images found in ground-truth folder: {GT_DIR}")

# Match raw <-> GT by filename (ignoring extension) so mismatched naming doesn't crash
raw_index = {os.path.splitext(os.path.basename(f))[0]: f for f in raw_files}
gt_index  = {os.path.splitext(os.path.basename(f))[0]: f for f in gt_files}

common_keys = sorted(set(raw_index.keys()) & set(gt_index.keys()))
missing_gt  = sorted(set(raw_index.keys()) - set(gt_index.keys()))
missing_raw = sorted(set(gt_index.keys()) - set(raw_index.keys()))

if missing_gt:
    print(f"WARNING: {len(missing_gt)} raw image(s) have no matching ground truth and "
          f"will be skipped for evaluation (but still enhanced), e.g.: {missing_gt[:5]}")
if missing_raw:
    print(f"WARNING: {len(missing_raw)} ground-truth image(s) have no matching raw "
          f"image and will be ignored, e.g.: {missing_raw[:5]}")

if len(common_keys) == 0:
    raise FileNotFoundError(
        "No filename matches at all between raw and ground-truth folders — please "
        "check that corresponding files share the same base filename."
    )

paired_files = [(raw_index[k], gt_index[k]) for k in common_keys]
print(f"Raw images found        : {len(raw_files)}")
print(f"Ground-truth images found: {len(gt_files)}")
print(f"Matched raw/GT pairs    : {len(paired_files)}")


In [ ]:
#@title Python re-implementation of Water-Net's preprocessing (white balance / gamma / HE)
# Faithful to testing_code_by_Branimir Ambrekovic/utils.py, adapted to modern
# OpenCV/NumPy (the original relies on the long-deprecated scipy.misc.imread/imresize).

def white_balance(img, percent=1):
    """
    Simple percentile color-stretch white balance, per channel.
    Matches the approach in Branimir Ambrekovic's utils.py (white_balance()).
    `img` is a uint8 BGR image.
    """
    out_channels = []
    h, w = img.shape[:2]
    cumstops = (
        h * w * percent / 200.0,
        h * w * (1 - percent / 200.0),
    )
    for channel in cv2.split(img):
        cumhist = np.cumsum(cv2.calcHist([channel], [0], None, [256], (0, 256)))
        low_cut, high_cut = np.searchsorted(cumhist, cumstops)
        high_cut = max(high_cut, low_cut + 1)
        lut = np.concatenate((
            np.zeros(low_cut),
            np.around(np.linspace(0, 255, high_cut - low_cut + 1)),
            255 * np.ones(255 - high_cut),
        ))
        lut = np.clip(lut, 0, 255).astype('uint8')
        out_channels.append(cv2.LUT(channel, lut))
    return cv2.merge(out_channels)


def adjust_gamma(image, gamma=0.7):
    """
    Gamma correction via lookup table. Matches Branimir Ambrekovic's utils.py
    (adjust_gamma()), default gamma=0.7.
    """
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255
                       for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image.astype(np.uint8), table)


def histogram_equalize(image):
    """
    Contrast enhancement via histogram equalization on the luminance channel only
    (YCrCb color space), which avoids the color-shift artifacts that per-channel RGB
    equalization would introduce. This produces Water-Net's third required input
    ("ce" / histogram-equalized branch).
    """
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    y, cr, cb = cv2.split(ycrcb)
    y_eq = cv2.equalizeHist(y)
    merged = cv2.merge([y_eq, cr, cb])
    return cv2.cvtColor(merged, cv2.COLOR_YCrCb2BGR)


def load_bgr(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Failed to read image (corrupt or unsupported format): {path}")
    return img

print("Preprocessing functions ready: white_balance(), adjust_gamma(), histogram_equalize()")


In [ ]:
#@title Generate the WB / GC / HE working copies for every raw image (fully automatic)
WORK_DIR = "/content/_wn_work"
WB_DIR = os.path.join(WORK_DIR, "wb_input")
GC_DIR = os.path.join(WORK_DIR, "gc_input")
CE_DIR = os.path.join(WORK_DIR, "ce_input")

for d in (WB_DIR, GC_DIR, CE_DIR):
    os.makedirs(d, exist_ok=True)

skipped = []
for raw_path, _gt_path in tqdm(paired_files, desc="Preprocessing (WB / Gamma / HE)"):
    fname = os.path.basename(raw_path)
    stem, _ = os.path.splitext(fname)
    out_name = stem + ".png"
    try:
        img = load_bgr(raw_path)
        wb_img = white_balance(img)
        gc_img = adjust_gamma(img)
        ce_img = histogram_equalize(img)
        cv2.imwrite(os.path.join(WB_DIR, out_name), wb_img)
        cv2.imwrite(os.path.join(GC_DIR, out_name), gc_img)
        cv2.imwrite(os.path.join(CE_DIR, out_name), ce_img)
    except Exception as e:
        print(f"Skipping '{fname}' due to preprocessing error: {e}")
        skipped.append(fname)

if skipped:
    paired_files = [(r, g) for (r, g) in paired_files
                     if os.path.basename(r) not in skipped]

print(f"Preprocessing complete. {len(paired_files)} images ready for inference "
      f"({len(skipped)} skipped due to errors).")


## Section 7 — Build the Water-Net graph and load the pretrained checkpoint

This reproduces the exact architecture from the official `model.py` (`T_CNN.model()`,
scope `main_branch`, layers `conv2wb_1` … `conv2wb_1111`) so that variable names line up
exactly with your checkpoint. It is written directly against `tf.compat.v1`
(`tf.compat.v1.get_variable`, `tf.compat.v1.placeholder`, `tf.compat.v1.train.Saver`,
etc.) so it runs unmodified on current Colab TensorFlow 2.x — this is the automatic
"TensorFlow compatibility mode" required by the project spec. Because every layer is a
stride-1, `SAME`-padded convolution, the network is fully convolutional and accepts
images of any height/width, so we build the graph **once** with a dynamic shape and
reuse the same session for every image (much faster than rebuilding per image).

In [ ]:
#@title Water-Net op primitives (tf.compat.v1, matches Branimir Ambrekovic's ops.py)
def conv2d(input_, output_dim, k_h=5, k_w=5, d_h=2, d_w=2, stddev=0.02, name="conv2d"):
    with tf.compat.v1.variable_scope(name):
        w = tf.compat.v1.get_variable(
            'w', [k_h, k_w, input_.get_shape()[-1], output_dim],
            initializer=tf.compat.v1.truncated_normal_initializer(stddev=stddev))
        conv = tf.nn.conv2d(input=input_, filters=w, strides=[1, d_h, d_w, 1], padding='SAME')
        biases = tf.compat.v1.get_variable(
            'biases', [output_dim], initializer=tf.compat.v1.constant_initializer(0.0))
        conv = tf.nn.bias_add(conv, biases)
        return conv

print("conv2d primitive ready.")


In [ ]:
#@title Water-Net T_CNN model definition (faithful port of model.py -> tf.compat.v1)
class WaterNetTF1(object):
    """
    Faithful TF2-compatible port of the official Water-Net T_CNN class
    (Li-Chongyi/Water-Net_Code/model.py), restricted to inference (no training code).
    Accepts dynamically-shaped [1, H, W, 3] inputs since the network is fully
    convolutional.
    """
    def __init__(self, sess, checkpoint_dir, label_height=112, c_dim=3):
        self.sess = sess
        self.checkpoint_dir = checkpoint_dir
        self.label_height = label_height   # used ONLY to build the "coarse_112" folder name
        self.c_dim = c_dim
        self._build_graph()

    def _build_graph(self):
        shape = [1, None, None, self.c_dim]
        self.images    = tf.compat.v1.placeholder(tf.float32, shape, name='images')
        self.images_wb = tf.compat.v1.placeholder(tf.float32, shape, name='images_wb')
        self.images_ce = tf.compat.v1.placeholder(tf.float32, shape, name='images_ce')
        self.images_gc = tf.compat.v1.placeholder(tf.float32, shape, name='images_gc')
        self.pred_h = self._model()
        self.saver = tf.compat.v1.train.Saver()

    def _model(self):
        with tf.compat.v1.variable_scope("main_branch"):
            conb0 = tf.concat(axis=3, values=[self.images, self.images_wb,
                                               self.images_ce, self.images_gc])
            conv_wb1 = tf.nn.relu(conv2d(conb0, 128, k_h=7, k_w=7, d_h=1, d_w=1, name="conv2wb_1"))
            conv_wb2 = tf.nn.relu(conv2d(conv_wb1, 128, k_h=5, k_w=5, d_h=1, d_w=1, name="conv2wb_2"))
            conv_wb3 = tf.nn.relu(conv2d(conv_wb2, 128, k_h=3, k_w=3, d_h=1, d_w=1, name="conv2wb_3"))
            conv_wb4 = tf.nn.relu(conv2d(conv_wb3, 64, k_h=1, k_w=1, d_h=1, d_w=1, name="conv2wb_4"))
            conv_wb5 = tf.nn.relu(conv2d(conv_wb4, 64, k_h=7, k_w=7, d_h=1, d_w=1, name="conv2wb_5"))
            conv_wb6 = tf.nn.relu(conv2d(conv_wb5, 64, k_h=5, k_w=5, d_h=1, d_w=1, name="conv2wb_6"))
            conv_wb7 = tf.nn.relu(conv2d(conv_wb6, 64, k_h=3, k_w=3, d_h=1, d_w=1, name="conv2wb_7"))
            conv_wb77 = tf.nn.sigmoid(conv2d(conv_wb7, 3, k_h=3, k_w=3, d_h=1, d_w=1, name="conv2wb_77"))

            conb00 = tf.concat(axis=3, values=[self.images, self.images_wb])
            conv_wb9  = tf.nn.relu(conv2d(conb00, 32, k_h=7, k_w=7, d_h=1, d_w=1, name="conv2wb_9"))
            conv_wb10 = tf.nn.relu(conv2d(conv_wb9, 32, k_h=5, k_w=5, d_h=1, d_w=1, name="conv2wb_10"))
            wb1 = tf.nn.relu(conv2d(conv_wb10, 3, k_h=3, k_w=3, d_h=1, d_w=1, name="conv2wb_11"))

            conb11 = tf.concat(axis=3, values=[self.images, self.images_ce])
            conv_wb99  = tf.nn.relu(conv2d(conb11, 32, k_h=7, k_w=7, d_h=1, d_w=1, name="conv2wb_99"))
            conv_wb100 = tf.nn.relu(conv2d(conv_wb99, 32, k_h=5, k_w=5, d_h=1, d_w=1, name="conv2wb_100"))
            ce1 = tf.nn.relu(conv2d(conv_wb100, 3, k_h=3, k_w=3, d_h=1, d_w=1, name="conv2wb_111"))

            conb111 = tf.concat(axis=3, values=[self.images, self.images_gc])
            conv_wb999  = tf.nn.relu(conv2d(conb111, 32, k_h=7, k_w=7, d_h=1, d_w=1, name="conv2wb_999"))
            conv_wb1000 = tf.nn.relu(conv2d(conv_wb999, 32, k_h=5, k_w=5, d_h=1, d_w=1, name="conv2wb_1000"))
            gc1 = tf.nn.relu(conv2d(conv_wb1000, 3, k_h=3, k_w=3, d_h=1, d_w=1, name="conv2wb_1111"))

            weight_wb, weight_ce, weight_gc = tf.split(conv_wb77, 3, axis=3)
            output = tf.add(tf.add(tf.multiply(wb1, weight_wb), tf.multiply(ce1, weight_ce)),
                             tf.multiply(gc1, weight_gc))
            return output

    def load_checkpoint(self):
        model_dir = "coarse_%s" % self.label_height
        coarse_dir = os.path.join(self.checkpoint_dir, model_dir)
        ckpt = tf.compat.v1.train.get_checkpoint_state(coarse_dir)
        if ckpt and ckpt.model_checkpoint_path:
            ckpt_name = os.path.basename(ckpt.model_checkpoint_path)
            self.saver.restore(self.sess, os.path.join(coarse_dir, ckpt_name))
            return True
        return False

    def enhance(self, raw01, wb01, ce01, gc01):
        """All inputs are HxWx3 float32 arrays scaled to [0, 1]."""
        feed = {
            self.images:    raw01[np.newaxis, ...],
            self.images_wb: wb01[np.newaxis, ...],
            self.images_ce: ce01[np.newaxis, ...],
            self.images_gc: gc01[np.newaxis, ...],
        }
        out = self.sess.run(self.pred_h, feed_dict=feed)
        return np.clip(out[0], 0.0, 1.0)

print("WaterNetTF1 class defined.")


In [ ]:
#@title Build the graph and restore the pretrained checkpoint
tf.compat.v1.reset_default_graph()
tf_config = tf.compat.v1.ConfigProto()
tf_config.gpu_options.allow_growth = True
sess = tf.compat.v1.Session(config=tf_config)

net = WaterNetTF1(sess, checkpoint_dir=CHECKPOINT_DIR, label_height=112)
sess.run(tf.compat.v1.global_variables_initializer())

loaded = net.load_checkpoint()
if not loaded:
    raise RuntimeError(
        f"Failed to load checkpoint from '{os.path.join(CHECKPOINT_DIR, 'coarse_112')}'. "
        f"Verify the checkpoint files are intact and match this architecture."
    )

print(" [*] Checkpoint loaded successfully — Water-Net is ready for inference.")


## Section 8 — Run batched inference over the whole dataset

In [ ]:
#@title Enhance every image and save results to Google Drive
def bgr_uint8_to_rgb01(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

enhanced_records = []   # list of dicts: name, raw_path, gt_path, output_path
failed_records = []

for raw_path, gt_path in tqdm(paired_files, desc="Running Water-Net inference"):
    fname = os.path.basename(raw_path)
    stem, _ = os.path.splitext(fname)
    out_name = stem + ".png"
    try:
        raw_bgr = load_bgr(raw_path)
        wb_bgr  = load_bgr(os.path.join(WB_DIR, out_name))
        gc_bgr  = load_bgr(os.path.join(GC_DIR, out_name))
        ce_bgr  = load_bgr(os.path.join(CE_DIR, out_name))

        raw01 = bgr_uint8_to_rgb01(raw_bgr)
        wb01  = bgr_uint8_to_rgb01(wb_bgr)
        gc01  = bgr_uint8_to_rgb01(gc_bgr)
        ce01  = bgr_uint8_to_rgb01(ce_bgr)

        enhanced01 = net.enhance(raw01, wb01, ce01, gc01)   # RGB, [0,1]

        out_path = os.path.join(RESULTS_DIR, out_name)
        enhanced_bgr = cv2.cvtColor((enhanced01 * 255.0).astype(np.uint8), cv2.COLOR_RGB2BGR)
        cv2.imwrite(out_path, enhanced_bgr)

        enhanced_records.append({
            "name": stem,
            "raw_path": raw_path,
            "gt_path": gt_path,
            "output_path": out_path,
        })
    except Exception as e:
        print(f"Inference failed on '{fname}': {e}")
        traceback.print_exc()
        failed_records.append(fname)

print(f"\nInference complete: {len(enhanced_records)} images enhanced and saved to "
      f"{RESULTS_DIR}")
if failed_records:
    print(f"{len(failed_records)} image(s) failed during inference: {failed_records}")


## Section 9 — Visualize Raw → Enhanced → Ground-Truth

In [ ]:
#@title Show side-by-side comparisons for a handful of sample images
N_SAMPLES = min(5, len(enhanced_records))
sample_records = enhanced_records[:N_SAMPLES]

if N_SAMPLES == 0:
    print("No successfully enhanced images available to visualize.")
else:
    fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(12, 4 * N_SAMPLES))
    if N_SAMPLES == 1:
        axes = axes[np.newaxis, :]

    for row, rec in enumerate(sample_records):
        raw_img = cv2.cvtColor(load_bgr(rec["raw_path"]), cv2.COLOR_BGR2RGB)
        out_img = cv2.cvtColor(load_bgr(rec["output_path"]), cv2.COLOR_BGR2RGB)
        gt_img  = cv2.cvtColor(load_bgr(rec["gt_path"]), cv2.COLOR_BGR2RGB)

        for col, (img, title) in enumerate(zip(
                [raw_img, out_img, gt_img],
                ["Raw", "Water-Net Enhanced", "Ground Truth"])):
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"{rec['name']} — {title}", fontsize=10)
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()


## Section 10 — Evaluate against ground truth (PSNR, SSIM, UIQM, UCIQE)

- **PSNR / SSIM** — full-reference metrics computed against `Dataset/ce_real` (your
  ground truth).
- **UIQM** (Underwater Image Quality Measure, Panetta et al., 2016) and **UCIQE**
  (Underwater Color Image Quality Evaluation, Yang & Sowmya, 2015) — no-reference
  metrics computed directly on the enhanced output, standard in underwater image
  enhancement literature.

In [ ]:
#@title No-reference UIQM / UCIQE metric implementations
def _uicm(img_rgb):
    R = img_rgb[..., 0].astype(np.float64)
    G = img_rgb[..., 1].astype(np.float64)
    B = img_rgb[..., 2].astype(np.float64)
    RG = R - G
    YB = 0.5 * (R + G) - B

    def _mu_alpha(x, alpha=0.1):
        x = np.sort(x.flatten())
        n = len(x)
        low = int(np.floor(alpha * n))
        high = int(np.ceil((1 - alpha) * n))
        trimmed = x[low:high] if high > low else x
        return trimmed.mean() if trimmed.size else x.mean()

    rg_mu, rg_var = _mu_alpha(RG), np.var(RG)
    yb_mu, yb_var = _mu_alpha(YB), np.var(YB)
    uicm = -0.0268 * math.sqrt(rg_mu ** 2 + yb_mu ** 2) + 0.1586 * math.sqrt(rg_var + yb_var)
    return uicm


def _uism(img_rgb):
    weights = (0.299, 0.587, 0.114)
    sm_total = 0.0
    for c in range(3):
        channel = img_rgb[..., c].astype(np.float64)
        sobelx = cv2.Sobel(channel, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(channel, cv2.CV_64F, 0, 1, ksize=3)
        edge_map = np.sqrt(sobelx ** 2 + sobely ** 2)
        sm_total += weights[c] * edge_map.mean()
    return sm_total


def _uiconm(img_rgb, block_size=8):
    gray = cv2.cvtColor(img_rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
    h, w = gray.shape
    logs = []
    for y in range(0, h - h % block_size, block_size):
        for x in range(0, w - w % block_size, block_size):
            block = gray[y:y + block_size, x:x + block_size]
            mx, mn = block.max(), block.min()
            if mx + mn > 0:
                contrast = (mx - mn) / (mx + mn + 1e-8)
                logs.append(contrast)
    if not logs:
        return 0.0
    logs = np.clip(logs, 1e-8, None)
    return float(np.mean(np.log(logs)))


def compute_uiqm(img_rgb_uint8):
    c1, c2, c3 = 0.0282, 0.2953, 3.5753
    uicm = _uicm(img_rgb_uint8)
    uism = _uism(img_rgb_uint8)
    uiconm = _uiconm(img_rgb_uint8)
    return float(c1 * uicm + c2 * uism + c3 * uiconm)


def compute_uciqe(img_rgb_uint8):
    lab = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2LAB).astype(np.float64)
    L, a, b = lab[..., 0], lab[..., 1], lab[..., 2]
    chroma = np.sqrt(a ** 2 + b ** 2)
    sigma_c = chroma.std()

    l_flat = np.sort(L.flatten())
    n = len(l_flat)
    top = l_flat[int(0.99 * n):] if int(0.99 * n) < n else l_flat[-1:]
    bottom = l_flat[:max(int(0.01 * n), 1)]
    con_l = (top.mean() if top.size else L.max()) - (bottom.mean() if bottom.size else L.min())

    hsv = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2HSV).astype(np.float64)
    mu_s = hsv[..., 1].mean()

    c1, c2, c3 = 0.4680, 0.2745, 0.2576
    return float(c1 * sigma_c + c2 * con_l + c3 * mu_s)

print("UIQM / UCIQE implementations ready.")


In [ ]:
#@title Compute metrics for every image and build the results table
rows = []
for rec in tqdm(enhanced_records, desc="Evaluating metrics"):
    try:
        enh_rgb = cv2.cvtColor(load_bgr(rec["output_path"]), cv2.COLOR_BGR2RGB)
        gt_rgb  = cv2.cvtColor(load_bgr(rec["gt_path"]), cv2.COLOR_BGR2RGB)

        if enh_rgb.shape != gt_rgb.shape:
            gt_rgb = cv2.resize(gt_rgb, (enh_rgb.shape[1], enh_rgb.shape[0]),
                                 interpolation=cv2.INTER_AREA)

        psnr_val = sk_psnr(gt_rgb, enh_rgb, data_range=255)
        ssim_val = sk_ssim(gt_rgb, enh_rgb, channel_axis=2, data_range=255)
        uiqm_val = compute_uiqm(enh_rgb)
        uciqe_val = compute_uciqe(enh_rgb)

        rows.append({
            "image": rec["name"],
            "PSNR": psnr_val,
            "SSIM": ssim_val,
            "UIQM": uiqm_val,
            "UCIQE": uciqe_val,
        })
    except Exception as e:
        print(f"Metric computation failed for '{rec['name']}': {e}")

metrics_df = pd.DataFrame(rows)

if metrics_df.empty:
    raise RuntimeError("No metrics could be computed — check that enhanced/GT images "
                        "loaded correctly in the previous sections.")

display_df = metrics_df.copy()
for col in ["PSNR", "SSIM", "UIQM", "UCIQE"]:
    display_df[col] = display_df[col].round(4)

print(display_df.to_string(index=False))


In [ ]:
#@title Export metrics.csv and metrics.xlsx to Google Drive
csv_path = os.path.join(RESULTS_DIR, "metrics.csv")
xlsx_path = os.path.join(RESULTS_DIR, "metrics.xlsx")

metrics_df.to_csv(csv_path, index=False)

summary_row = {
    "image": "AVERAGE",
    "PSNR": metrics_df["PSNR"].mean(),
    "SSIM": metrics_df["SSIM"].mean(),
    "UIQM": metrics_df["UIQM"].mean(),
    "UCIQE": metrics_df["UCIQE"].mean(),
}
export_df = pd.concat([metrics_df, pd.DataFrame([summary_row])], ignore_index=True)
export_df.to_excel(xlsx_path, index=False)

print("Exported:")
print(" -", csv_path)
print(" -", xlsx_path)


## Section 11 — Summary

In [ ]:
#@title Final run summary
avg_psnr = metrics_df["PSNR"].mean()
avg_ssim = metrics_df["SSIM"].mean()
avg_uiqm = metrics_df["UIQM"].mean()
avg_uciqe = metrics_df["UCIQE"].mean()

print("=" * 55)
print(" WATER-NET INFERENCE & BENCHMARKING — SUMMARY")
print("=" * 55)
print(f" Images processed successfully : {len(enhanced_records)}")
print(f" Images failed                 : {len(failed_records)}")
print(f" Average PSNR                  : {avg_psnr:.4f} dB")
print(f" Average SSIM                  : {avg_ssim:.4f}")
print(f" Average UIQM                  : {avg_uiqm:.4f}")
print(f" Average UCIQE                 : {avg_uciqe:.4f}")
print("-" * 55)
print(f" Enhanced images  : {RESULTS_DIR}")
print(f" metrics.csv      : {csv_path}")
print(f" metrics.xlsx     : {xlsx_path}")
print("=" * 55)
